In [10]:
"""
Sankey Plot: Design Principle → Descriptive Codes → Student Reactions
Data source: db-q4.csv
Requirements: pip install pandas plotly kaleido
"""

import pandas as pd
import plotly.graph_objects as go

# int_var2 = "Descriptive Codes"

int_var = "Descriptive Codes"
# int_var = "Exact question"


# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv("db-4-5.csv")
df.columns = df.columns.str.strip()
df = df.dropna(subset=["Design Principle", int_var, "Students reactions"])

# ── 2. Build node list (order: Design Principles → Descriptive Codes → Reactions)
design_principles = sorted(df["Design Principle"].unique().tolist())
descriptive_codes = sorted(df[int_var].unique().tolist())
student_reactions = sorted(df["Students reactions"].unique().tolist())

all_nodes = design_principles + descriptive_codes + student_reactions

node_index = {name: i for i, name in enumerate(all_nodes)}

dp_count  = len(design_principles)
dc_count  = len(descriptive_codes)
sr_count  = len(student_reactions)

In [ ]:
## OFFICIAL

# ── 2b. Select 20 codes: top-10 most popular + 10 most unique ─────────────────
dc_counts = df[int_var].value_counts()
top10_popular = dc_counts.nlargest(10).index.tolist()
remaining_counts = dc_counts.drop(top10_popular)
top10_unique = remaining_counts.nsmallest(10).index.tolist()
selected_codes = set(top10_popular + top10_unique)

# ── Visibility config ─────────────────────────────────────────────────────────
SHOW_MIN    = 2      # links with value <= this are invisible (but count toward node size)
ALWAYS_SHOW = ["team", "teamwork", "collaborat"]          # always draw these links
ALWAYS_HIDE = ["system structure easy to navigate"]       # remove these nodes entirely

def matches(label, keywords):
    return any(kw in label.lower() for kw in keywords)

# ── 2c. Rebuild node lists excluding hidden nodes ─────────────────────────────
vis_dp = [n for n in design_principles if not matches(n, ALWAYS_HIDE)]
vis_dc = [n for n in descriptive_codes  if not matches(n, ALWAYS_HIDE)]
vis_sr = [n for n in student_reactions  if not matches(n, ALWAYS_HIDE)]

all_nodes_vis = vis_dp + vis_dc + vis_sr
node_index_vis = {name: i for i, name in enumerate(all_nodes_vis)}

vdp, vdc, vsr = len(vis_dp), len(vis_dc), len(vis_sr)

# ── 3. Build links (ALL data, skip links touching hidden nodes) ───────────────
layer1 = (
    df.groupby(["Design Principle", int_var])
    .size()
    .reset_index(name="value")
)

layer2 = (
    df.groupby([int_var, "Students reactions"])
    .size()
    .reset_index(name="value")
)

sources, targets, values = [], [], []

for _, row in layer1.iterrows():
    a, b = row["Design Principle"], row[int_var]
    if a not in node_index_vis or b not in node_index_vis:
        continue
    sources.append(node_index_vis[a])
    targets.append(node_index_vis[b])
    values.append(row["value"])

for _, row in layer2.iterrows():
    a, b = row[int_var], row["Students reactions"]
    if a not in node_index_vis or b not in node_index_vis:
        continue
    sources.append(node_index_vis[a])
    targets.append(node_index_vis[b])
    values.append(row["value"])

# ── 4. Colour palettes ────────────────────────────────────────────────────────
dp_color_list = ["#4C72B0", "#8172B2", "#55A868", "#5A8F7B", "#DD8452"]
node_colors = dp_color_list[:vdp] + ["#A8C8E8"] * vdc + [
    "#E07B54", "#6BAED6", "#74C476", "#9E9AC8",
    "#FD8D3C", "#FDAE6B", "#31A354", "#756BB1", "#636363"
][:vsr]

# ── 4b. Labels ────────────────────────────────────────────────────────────────
dc_labels  = [code if code in selected_codes else "" for code in vis_dc]
all_labels = vis_dp + dc_labels + vis_sr

# ── 4c. Link colours ─────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, alpha=0.35):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(node_colors[s])
    if (v > SHOW_MIN or matches(all_labels[s], ALWAYS_SHOW) or matches(all_labels[t], ALWAYS_SHOW))
    else "rgba(0,0,0,0)"
    for s, t, v in zip(sources, targets, values)
]

# ── 4d. Node x/y positions ───────────────────────────────────────────────────
x_dp, x_dc, x_sr = 0.01, 2/6, 0.99

def col_y(n):
    if n == 1:
        return [0.5]
    return [0.02 + i * 0.96 / (n - 1) for i in range(n)]

node_x = [x_dp] * vdp + [x_dc] * vdc + [x_sr] * vsr
node_y = col_y(vdp) + col_y(vdc) + col_y(vsr)

# ── 5. Build figure ───────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=8,
        thickness=14,
        line=dict(color="white", width=0.5),
        label=all_labels,
        color=node_colors,
        x=node_x,
        y=node_y,
        hovertemplate="<b>%{label}</b><br>Total flow: %{value}<extra></extra>",
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b>"
            "<br>Count: %{value}<extra></extra>"
        ),
    ),
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Design Principle → Descriptive Codes → Student Reactions</b>"
            "<br><sup>Each band width is proportional to the number of observations</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    font=dict(family="Arial", size=16, color="#333333"),
    paper_bgcolor="white",
    height=450,
    width=1400,
    margin=dict(l=20, r=20, t=80, b=10),
)

# ── 6. Layer label annotations ────────────────────────────────────────────────
for x_pos, label in zip([0.01, 1/6, 0.99], ["Design Principles", int_var, "Student Reactions"]):
    fig.add_annotation(
        x=x_pos, y=1.06,
        xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=20, color="#444"),
        align="center",
    )

# ── 7. Save outputs ───────────────────────────────────────────────────────────
fig.write_html("sankey_output.html")
print("✅  Saved: sankey_output.html  (interactive)")
print(f"   Hidden nodes: {ALWAYS_HIDE}")

try:
    fig.write_image("sankey_output.png", scale=2)
    print("✅  Saved: sankey_output.png   (static, 2× resolution)")
except Exception as e:
    print(f"⚠️  PNG export skipped ({e}). Install kaleido: pip install kaleido")

fig.show()


✅  Saved: sankey_output.html  (interactive)
⚠️  PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Install kaleido: pip install kaleido


In [2]:
"""
Sankey Plot: Design Principle → Exact Question → Descriptive Codes → Student Reactions
Data source: db-q4-2.csv
Requirements: pip install pandas plotly kaleido
"""

import pandas as pd
import plotly.graph_objects as go

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv("db-q4-2.csv")
df.columns = df.columns.str.strip()
df = df.dropna(subset=["Design Principle", "Exact question", "Descriptive Codes", "Students reactions"])

# ── 2. Build node list (4 layers in order) ───────────────────────────────────
design_principles = sorted(df["Design Principle"].unique().tolist())
exact_questions   = sorted(df["Exact question"].unique().tolist())
descriptive_codes = sorted(df["Descriptive Codes"].unique().tolist())
student_reactions = sorted(df["Students reactions"].unique().tolist())

all_nodes = design_principles + exact_questions + descriptive_codes + student_reactions

node_index = {name: i for i, name in enumerate(all_nodes)}

dp_count = len(design_principles)
eq_count = len(exact_questions)
dc_count = len(descriptive_codes)
sr_count = len(student_reactions)

# ── 3. Build links (3 link layers) ───────────────────────────────────────────
# Layer 1 → 2 : Design Principle → Exact Question
layer1 = (
    df.groupby(["Design Principle", "Exact question"])
    .size()
    .reset_index(name="value")
)

# Layer 2 → 3 : Exact Question → Descriptive Codes
layer2 = (
    df.groupby(["Exact question", "Descriptive Codes"])
    .size()
    .reset_index(name="value")
)

# Layer 3 → 4 : Descriptive Codes → Student Reactions
layer3 = (
    df.groupby(["Descriptive Codes", "Students reactions"])
    .size()
    .reset_index(name="value")
)

sources, targets, values = [], [], []

for _, row in layer1.iterrows():
    sources.append(node_index[row["Design Principle"]])
    targets.append(node_index[row["Exact question"]])
    values.append(row["value"])

for _, row in layer2.iterrows():
    sources.append(node_index[row["Exact question"]])
    targets.append(node_index[row["Descriptive Codes"]])
    values.append(row["value"])

for _, row in layer3.iterrows():
    sources.append(node_index[row["Descriptive Codes"]])
    targets.append(node_index[row["Students reactions"]])
    values.append(row["value"])

# ── 4. Colour palettes ────────────────────────────────────────────────────────
dp_colors = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"
][:dp_count]

eq_colors = ["#B5CEED"] * eq_count      # light blue for Exact Questions

dc_colors = ["#A8C8E8"] * dc_count      # slightly different blue for Descriptive Codes

sr_colors = [
    "#E07B54", "#6BAED6", "#74C476", "#9E9AC8",
    "#FD8D3C", "#FDAE6B", "#31A354", "#756BB1", "#636363"
][:sr_count]

node_colors = dp_colors + eq_colors + dc_colors + sr_colors

# Link colours: inherit source node colour with transparency
def hex_to_rgba(hex_color, alpha=0.35):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(node_colors[s]) for s in sources]

# ── 5. Build figure ───────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=22,
        line=dict(color="white", width=0.5),
        label=all_nodes,
        color=node_colors,
        hovertemplate="<b>%{label}</b><br>Total flow: %{value}<extra></extra>",
    ),
    
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b>"
            "<br>Count: %{value}<extra></extra>"
        ),
    ),
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Design Principle → Exact Question → Descriptive Codes → Student Reactions</b>"
            "<br><sup>Each band width is proportional to the number of observations</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=16),
    ),
    font=dict(family="Arial", size=11, color="#333333"),
    paper_bgcolor="white",
    height=950,
    width=1600,
    margin=dict(l=20, r=20, t=90, b=20),
)

# ── 6. Layer label annotations ────────────────────────────────────────────────
layer_labels = ["Design Principles", "Exact Question", "Descriptive Codes", "Student Reactions"]
x_positions  = [0.01, 0.34, 0.67, 0.99]

for x_pos, label in zip(x_positions, layer_labels):
    fig.add_annotation(
        x=x_pos, y=1.04,
        xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=13, color="#444"),
        align="center",
    )

# ── 7. Save outputs ───────────────────────────────────────────────────────────
fig.write_html("sankey_output.html")
print("✅  Saved: sankey_output.html  (interactive)")

try:
    fig.write_image("sankey_output.png", scale=2)
    print("✅  Saved: sankey_output.png   (static, 2× resolution)")
except Exception as e:
    print(f"⚠️  PNG export skipped ({e}). Install kaleido: pip install kaleido")

fig.show()

✅  Saved: sankey_output.html  (interactive)
⚠️  PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Install kaleido: pip install kaleido


In [ ]:
t